# 1. Objective and Setup

This notebook uses information from the Hidden Markov Model (HMM) to improve one-step-ahead volatility forecasts. At each step, only information available at that time is used. The forecasts will later be used for Value-at-Risk (VaR) backtesting.

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    

In [3]:
from src import data_loader as dl
from src import preprocessing as pp
from src import garch
from src import features
from src.markov_model.hmm import HiddenMarkovModel as hmm
from src.markov_model.forward import forward_filtering
from src.markov_model.backward import backward_filtering
from sklearn.preprocessing import StandardScaler

# Download S&P 500 data from 2000-01-01 and compute daily log returns.
raw_data = dl.load_market_data()
market_data = pp.preprocess_market_data(raw_data)
vix = dl.load_market_data(ticker = "^VIX")
garch_output = garch.fit_garch(market_data["Return"]*100)

In [11]:
baseline_features = features.build_baseline_features(market_data, garch_output)
alt_features = features.build_alternative_features(market_data, vix)
all_features = pd.concat([baseline_features, alt_features], axis=1)
final_features = all_features[["Momentum_20", "Conditional_Volatility", "VIX"]].dropna().sort_index()


## 2. Forecasting Dataset

The final HMM uses 20-day momentum, GARCH conditional volatility, and VIX. The data is split chronologically into 70% training and 30% test data to preserve the time-series structure.

The features are standardized using a StandardScaler fitted only on the training data, preventing information from the test period from entering the model. A three-state Gaussian HMM with full covariance matrices is then fitted to the training sample.

The test period is reserved for out-of-sample one-step-ahead forecasting, where only information available at each forecast origin is used to avoid look-ahead bias.

In [16]:
split_idx = int(len(final_features) *0.7)
df_train = final_features.iloc[:split_idx].copy()
df_test = final_features.iloc[split_idx:].copy()

scaler = StandardScaler()
X_train = scaler.fit_transform(df_train)
X_test = scaler.transform(df_test)

model = hmm(feature_matrix = X_train, states = 3, covariance_type="full").fit()

## 3. Regime-Specific Volatility Characteristics

Before constructing the forecasting model, the return characteristics of the identified regimes are examined. Smoothed probabilities are used only for this in-sample analysis and are not used for out-of-sample forecasting.

In [17]:
train_probs = model.smooth_proba(X_train)
train_states = np.argmax(train_probs, axis=1)

regime_analysis = df_train.copy()

regime_analysis["Returns"] = market_data.loc[
    df_train.index,
    "Return"
]

regime_analysis["State"] = train_states

regime_stats = regime_analysis.groupby("State")["Returns"].agg([
    "count",
    "mean",
    "std",
    "min",
    "max"
])

display(regime_stats)

,count,mean,std,min,max
State,,,,,
0,1948,-0.000003,0.010762,-0.036581,0.038183
1,700,-0.000749,0.023028,-0.094695,0.109572
2,2004,0.000593,0.005919,-0.023234,0.021336


The regimes exhibit clearly different return distributions. State 2 represents the calm regime, with positive average returns and the lowest volatility. State 1 represents the stressed regime, with negative average returns and substantially higher volatility, while State 0 represents an intermediate regime. In particular, return volatility in the stressed regime is almost four times higher than in the calm regime. These differences motivate investigating whether HMM regime information can improve volatility forecasts.

## 4. Baseline GARCH Forecasting

A standard GARCH(1,1) model with Student-t innovations is used as the baseline volatility model. The model is initially estimated using only the training sample. One-step-ahead volatility forecasts are then done for the out-of-sample period using information available at each timestep. These forecasts provide the benchmark against which the regime-based model will later be evaluated. The formula for forecasting is: $\sigma^2_{t+1|t} = \omega + \alpha \epsilon^2_t + \beta \sigma^2_t$. The forecast depends on the most recent squared shock and conditional variance, both of which are known at time $t$.

### 4.1 Initial Model Estimation

Model fitted on training returns.

In [25]:
train_returns = market_data.loc[df_train.index,"Return"]
test_returns = market_data.loc[df_test.index,"Return"]


baseline_garch_output = garch.fit_garch(train_returns * 100, distribution="t")
baseline_garch_model = baseline_garch_output.result
params = baseline_garch_model.params

print(params)

alpha = params["alpha[1]"]
beta = params["beta[1]"]

print(f"\nVolatility persistence (alpha + beta): "f"{alpha + beta:.4f}")

mu          0.066078
omega       0.009561
alpha[1]    0.103372
beta[1]     0.894339
nu          6.556882
Name: params, dtype: float64

Volatility persistence (alpha + beta): 0.9977


#### Discussion

The baseline GARCH(1,1) model shows high volatility persistence, with $\alpha + \beta = 0.9977$. This means that changes in volatility tend to persist over time. The Student-t parameter of 6.56 also suggests that extreme returns occur more often than under a normal distribution.

### 4.2 Walking-Forward Volatility Forecasts

One-step-ahead volatility forecasts are generated in sequence throughout the test period. The GARCH conditional variance is updated each day as new with new returns, while the model parameters are re-estimated at the beginning of each month using an expanding historical window. This approach allows the model to adapt over time without requiring a full re-estimation every trading day.